In [ ]:
# Was run on kaggle under URL: https://www.kaggle.com/code/artemchistyakov/catboost-symmtree-comparison

!pip install -q openml --upgrade # to counter "dataset not found"

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.4/160.4 kB 4.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.8/93.8 kB 4.9 MB/s eta 0:00:00


In [2]:
"""
Benchmark: CatBoost Vanilla vs Experimental
============================================
Two dataset sources, unified evaluation pipeline (TabM-style):
  - OpenML  : downloaded via fetch_openml, split into train/val/test once
  - TabM    : pre-split datasets from disk

Variants
--------
CB_Default         : stock CatBoost, Depthwise, depth=6
CB_Default_depth=9 : stock CatBoost, Depthwise, depth=9
CB_Experimental    : custom build,   Depthwise, depth=9, dev_efb_max_buckets=0

Google Drive sync  (rclone)
----------------------------
One-time setup — do this on your laptop BEFORE running on Kaggle:

  1. Install rclone:  https://rclone.org/install/
  2. Run:             rclone config
       → choose "n" (new remote)
       → name it exactly:  gdrive
       → type: "drive"
       → follow the browser auth flow
  3. After config is saved, run:
       rclone config show gdrive
     Copy the entire printed block, e.g.:
       [gdrive]
       type = drive
       client_id = ...
       token = {"access_token": "...", "refresh_token": "...", ...}
       ...
  4. In Kaggle → your notebook → "Add-ons" → "Secrets":
       Name:  RCLONE_CONF
       Value: <paste the block from step 3>
     Enable the secret for this notebook.

That's it. The script will auto-install rclone inside Kaggle, write the conf,
pull any existing CSV from Drive on startup, and push after every new row.
"""

# ── Standard library ──────────────────────────────────────────────────────────
import gc
import json
import os
import subprocess
import sys
import time
import warnings
from pathlib import Path

# ── Third-party ───────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.metrics import r2_score, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder

warnings.filterwarnings("ignore")

# =============================================================================
# CONFIGURATION
# =============================================================================

SEED             = 42
BASE_ITER        = 10000
BASE_LR          = 0.05
BASE_ESR         = 50
N_SEEDS          = 15

OPENML_VAL_FRAC  = 0.1
OPENML_TEST_FRAC = 0.1

# ── Paths ─────────────────────────────────────────────────────────────────────
# Local copy inside the Kaggle container (fast writes, lost if env dies)
RESULTS_CSV_LOCAL  = Path("/kaggle/working/benchmark_results.csv")

# Where to store on Google Drive
GDRIVE_REMOTE_NAME = "gdrive"                    # must match your rclone remote name
GDRIVE_REMOTE_DIR  = "benchmark"                 # folder that will be created in Drive root
GDRIVE_REMOTE_CSV  = (
    f"{GDRIVE_REMOTE_NAME}:{GDRIVE_REMOTE_DIR}/benchmark_results_symmtree.csv"
)

MODELS_DIR: Path | None = None   # set to a Path to save .cbm files; None = don't save

TABM_ROOT = Path(
    "/kaggle/input/datasets/artemchistyakov/datasets-in-tabm/datasets_original"
)
PATH_CUSTOM_CB = Path(
    "/kaggle/input/datasets/artemchistyakov/catboostdepthonly3-9/catboost/catboost/python-package"
)

TABM_SKIP: set[str] = {"covtype2", "microsoft", "otto"}

np.random.seed(SEED)

# =============================================================================
# RCLONE / GOOGLE DRIVE HELPERS
# =============================================================================

# Флаг выставляется в True после успешного _setup_rclone()
GDRIVE_READY: bool = False


def _normalize_rclone_conf(conf_text: str) -> str:
    """
    Kaggle Secrets сжимает переносы строк в пробелы.
    Восстанавливаем нормальный ini-формат:
      [gdrive] type = drive  →  [gdrive]\ntype = drive\n...
    """
    import re
    # Если переносы уже есть — ничего не делаем
    if "\n" in conf_text:
        return conf_text
    # Вставляем \n перед [section] и перед каждым "key = value"
    # Сначала перед секциями
    conf_text = re.sub(r'\s*(\[[^\]]+\])\s*', r'\n\1\n', conf_text)
    # Затем перед каждым "word = " (ключи ini)
    conf_text = re.sub(r'\s+([a-zA-Z_][a-zA-Z_0-9]*\s*=)', r'\n\1', conf_text)
    return conf_text.strip()


def _setup_rclone() -> bool:
    """
    1. Читает токен из переменной окружения RCLONE_CONF или из Kaggle Secrets.
    2. Восстанавливает переносы строк (Kaggle их съедает).
    3. Записывает конфиг в ~/.config/rclone/rclone.conf.
    4. Устанавливает rclone, если его нет.
    5. Делает smoke-test.
    Возвращает True если Drive готов к использованию.
    """
    # --- получить конфиг ---
    conf_text = os.environ.get("RCLONE_CONF", "").strip()
    if not conf_text:
        try:
            from kaggle_secrets import UserSecretsClient          # noqa
            conf_text = UserSecretsClient().get_secret("RCLONE_CONF").strip()
        except Exception:
            pass

    if not conf_text:
        print("⚠  RCLONE_CONF secret not found — Google Drive sync DISABLED.")
        print("   Results will only be saved to /kaggle/working/ (lost if env dies).")
        return False

    # --- восстановить переносы строк, съеденные Kaggle Secrets ---
    conf_text = _normalize_rclone_conf(conf_text)

    # --- записать конфиг ---
    conf_path = Path.home() / ".config" / "rclone" / "rclone.conf"
    conf_path.parent.mkdir(parents=True, exist_ok=True)
    conf_path.write_text(conf_text)
    print(f"  rclone.conf written to {conf_path}")

    # --- установить rclone если нет ---
    if subprocess.run(["which", "rclone"], capture_output=True).returncode != 0:
        print("  rclone not found — installing...")
        r = subprocess.run(
            "curl https://rclone.org/install.sh | sudo bash",
            shell=True, capture_output=True, text=True,
        )
        if r.returncode != 0:
            print(f"⚠  rclone install failed:\n{r.stderr}")
            return False
        print("  rclone installed.")

    # --- smoke-test: проверить что remote отвечает ---
    r = subprocess.run(
        ["rclone", "lsd", f"{GDRIVE_REMOTE_NAME}:"],
        capture_output=True, text=True,
    )
    if r.returncode != 0:
        print(f"⚠  rclone cannot reach remote '{GDRIVE_REMOTE_NAME}':\n{r.stderr.strip()}")
        print("   Check that the remote name in RCLONE_CONF matches GDRIVE_REMOTE_NAME.")
        return False

    print(f"✓  rclone connected to remote '{GDRIVE_REMOTE_NAME}'")
    return True


def _pull_csv_from_drive() -> None:
    """Скачать CSV с Drive в локальный путь (если файл там уже есть)."""
    r = subprocess.run(
        ["rclone", "copyto", GDRIVE_REMOTE_CSV, str(RESULTS_CSV_LOCAL)],
        capture_output=True, text=True,
    )
    if r.returncode == 0 and RESULTS_CSV_LOCAL.exists():
        rows = sum(1 for _ in open(RESULTS_CSV_LOCAL)) - 1  # минус заголовок
        print(f"✓  Pulled existing CSV from Drive ({rows} rows — will skip these).")
    else:
        print("  No existing CSV on Drive — starting fresh.")


def _push_csv_to_drive() -> None:
    """Загрузить локальный CSV на Drive (быстро, только изменённые байты)."""
    if not RESULTS_CSV_LOCAL.exists():
        return
    r = subprocess.run(
        ["rclone", "copyto", str(RESULTS_CSV_LOCAL), GDRIVE_REMOTE_CSV],
        capture_output=True, text=True,
    )
    if r.returncode != 0:
        print(f"⚠  Drive push failed: {r.stderr.strip()}")

# =============================================================================
# VARIANT DEFINITIONS
# =============================================================================

VARIANTS: dict[str, dict] = {
    "CB_Default": {
        "iterations":            BASE_ITER * 3,
        "learning_rate":         BASE_LR,
        "depth":                 6,
        "grow_policy":           "SymmetricTree",
        "early_stopping_rounds": BASE_ESR,
        "experimental":          False,
    },
    "CB_Default_depth=9": {
        "iterations":            BASE_ITER,
        "learning_rate":         BASE_LR,
        "depth":                 9,
        "grow_policy":           "SymmetricTree",
        "early_stopping_rounds": BASE_ESR,
        "experimental":          False,
    },
    "CB_Experimental": {
        "iterations":            BASE_ITER * 2,
        "learning_rate":         BASE_LR,
        "depth":                 9,
        "dev_efb_max_buckets":   0,
        "grow_policy":           "SymmetricTree",
        "early_stopping_rounds": BASE_ESR,
        "experimental":          True,
    },
}

# =============================================================================
# CATBOOST LOADER
# =============================================================================

def load_catboost(experimental: bool):
    for key in [k for k in sys.modules
                if k.startswith("catboost") or k.startswith("_catboost")]:
        sys.modules.pop(key)

    custom = str(PATH_CUSTOM_CB)
    if experimental:
        if custom not in sys.path:
            sys.path.insert(0, custom)
    else:
        if custom in sys.path:
            sys.path.remove(custom)

    import catboost
    print(f"    catboost: {catboost.__file__}")
    return catboost

# =============================================================================
# DATA LOADING
# =============================================================================

def load_openml_dataset(name: str, config: dict) -> dict | None:
    print(f"\n>>> [OpenML] Loading {name}...")
    try:
        raw  = fetch_openml(data_id=config["id"], as_frame=True, parser="auto")
        X, y = raw.data, raw.target

        if config.get("target") is not None and config["target"] in X.columns:
            y = X[config["target"]]
            X = X.drop(columns=[config["target"]])

        cat_cols = X.select_dtypes(include=["category", "object"]).columns.tolist()
        if cat_cols:
            enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
            X[cat_cols] = enc.fit_transform(X[cat_cols])
            X[cat_cols] = X[cat_cols].astype(np.int64)

        X = X.fillna(X.median(numeric_only=True))
        y = pd.to_numeric(y, errors="coerce").fillna(0).astype(float)

        mask = np.isfinite(y)
        X, y = X[mask].reset_index(drop=True), y[mask].reset_index(drop=True)

        X_train, X_tmp, y_train, y_tmp = train_test_split(
            X, y,
            test_size=OPENML_VAL_FRAC + OPENML_TEST_FRAC,
            random_state=SEED,
        )
        val_share = OPENML_VAL_FRAC / (OPENML_VAL_FRAC + OPENML_TEST_FRAC)
        X_val, X_test, y_val, y_test = train_test_split(
            X_tmp, y_tmp, test_size=1 - val_share, random_state=SEED,
        )

        cat_feature_indices = [X_train.columns.get_loc(c) for c in cat_cols]
        print(f"    Shape — train: {X_train.shape}  val: {X_val.shape}  test: {X_test.shape}")
        return dict(
            X_train=X_train,       y_train=y_train.values,
            X_val=X_val,           y_val=y_val.values,
            X_test=X_test,         y_test=y_test.values,
            cat_features=cat_feature_indices,
            task_type="regression",
        )
    except Exception as exc:
        print(f"!! Failed to load {name}: {exc}")
        return None


def load_tabm_dataset(name: str) -> dict:
    path = TABM_ROOT / name

    with open(path / "info.json") as f:
        info = json.load(f)
    task_type = info["task_type"]

    parts_train, parts_val, parts_test = [], [], []
    cat_features: list[int] = []
    col_cursor = 0

    for feat_type in ("num", "bin", "cat"):
        fpath = path / f"X_{feat_type}_train.npy"
        if not fpath.exists():
            continue

        tr = np.load(fpath,                             allow_pickle=True)
        va = np.load(path / f"X_{feat_type}_val.npy",  allow_pickle=True)
        te = np.load(path / f"X_{feat_type}_test.npy", allow_pickle=True)

        if feat_type == "cat":
            cat_features.extend(range(col_cursor, col_cursor + tr.shape[1]))
            for arr in (tr, va, te):
                try:
                    arr[:] = np.round(arr.astype(np.float64), 4)
                except (ValueError, TypeError):
                    pass

        parts_train.append(tr)
        parts_val.append(va)
        parts_test.append(te)
        col_cursor += tr.shape[1]

    def _to_df(parts):
        return pd.DataFrame(np.hstack(parts))

    X_train, X_val, X_test = _to_df(parts_train), _to_df(parts_val), _to_df(parts_test)

    for col in cat_features:
        for df in (X_train, X_val, X_test):
            try:
                df[col] = pd.to_numeric(df[col]).round().astype(np.int64)
            except Exception:
                df[col] = df[col].astype(object)

    return dict(
        X_train=X_train, y_train=np.load(path / "Y_train.npy"),
        X_val=X_val,     y_val=np.load(path / "Y_val.npy"),
        X_test=X_test,   y_test=np.load(path / "Y_test.npy"),
        cat_features=cat_features,
        task_type=task_type,
    )

# =============================================================================
# TRAINING & EVALUATION
# =============================================================================

def build_catboost_model(variant_cfg: dict, task_type: str, seed: int):
    cb     = load_catboost(variant_cfg["experimental"])
    params = {k: v for k, v in variant_cfg.items() if k != "experimental"}
    params.update({"random_seed": seed, "verbose": 0, "allow_writing_files": False})

    if task_type == "regression":
        params["loss_function"] = "RMSE"
        return cb.CatBoostRegressor(**params)
    elif task_type == "binclass":
        params["loss_function"] = "Logloss"
        return cb.CatBoostClassifier(**params)
    else:
        params["loss_function"] = "MultiClass"
        return cb.CatBoostClassifier(**params)


def train_model(model, data: dict) -> float:
    t0 = time.perf_counter()
    model.fit(
        data["X_train"], data["y_train"],
        eval_set=(data["X_val"], data["y_val"]),
        cat_features=data["cat_features"] or None,
        use_best_model=True,
    )
    return time.perf_counter() - t0


def compute_score(model, data: dict) -> float:
    task_type      = data["task_type"]
    X_test, y_test = data["X_test"], data["y_test"]
    if task_type == "regression":
        return float(r2_score(y_test, model.predict(X_test)))
    else:
        return float(accuracy_score(y_test, np.argmax(model.predict_proba(X_test), axis=1)))

# =============================================================================
# RESULT STREAMING & CHECKPOINTING
# =============================================================================

def append_row(row: dict) -> None:
    """Дописать одну строку в локальный CSV, затем сразу синхронизировать с Drive."""
    df           = pd.DataFrame([row])
    write_header = not RESULTS_CSV_LOCAL.exists()
    df.to_csv(
        RESULTS_CSV_LOCAL, mode="a", header=write_header,
        index=False, float_format="%.8f",
    )
    if GDRIVE_READY:
        _push_csv_to_drive()


def already_done(dataset: str, variant: str, seed: int) -> bool:
    """True если эта (dataset, variant, seed) тройка уже есть в CSV."""
    if not RESULTS_CSV_LOCAL.exists():
        return False
    df = pd.read_csv(RESULTS_CSV_LOCAL)
    return (
        (df["dataset"] == dataset) &
        (df["variant"] == variant) &
        (df["seed"]    == seed)
    ).any()


def model_ckpt_path(dataset: str, variant: str, seed: int) -> Path | None:
    if MODELS_DIR is None:
        return None
    return MODELS_DIR / dataset / variant / f"seed_{seed:02d}.cbm"


def save_model(model, dataset: str, variant: str, seed: int) -> None:
    path = model_ckpt_path(dataset, variant, seed)
    if path is None:
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    model.save_model(str(path))

# =============================================================================
# UNIFIED BENCHMARK LOOP
# =============================================================================

def run_benchmark(datasets: dict[str, dict | None], source: str) -> None:
    total = len(datasets) * len(VARIANTS) * N_SEEDS
    done  = 0

    print("\n" + "=" * 70)
    print(f"Source: {source.upper()}  |  datasets={len(datasets)}  "
          f"variants={len(VARIANTS)}  seeds={N_SEEDS}")
    print("=" * 70)

    for ds_name, ds_config in datasets.items():
        print(f"\n{'─' * 70}\nDataset: {ds_name}")

        if source == "openml":
            data = load_openml_dataset(ds_name, ds_config)
        else:
            try:
                data = load_tabm_dataset(ds_name)
            except Exception as exc:
                print(f"  ✗ Failed to load: {exc}")
                continue

        if data is None:
            continue

        metric_name = "R2" if data["task_type"] == "regression" else "Accuracy"
        print(f"  Task: {data['task_type']}  |  Metric: {metric_name}")
        print(f"  Shapes — train: {data['X_train'].shape}  "
              f"val: {data['X_val'].shape}  test: {data['X_test'].shape}")

        for variant_name, variant_cfg in VARIANTS.items():
            seed_scores: list[float] = []

            for seed in range(N_SEEDS):
                done += 1
                label = f"  [{done:4d}/{total}] {variant_name:20s} seed={seed:2d}"

                ckpt       = model_ckpt_path(ds_name, variant_name, seed)
                csv_done   = already_done(ds_name, variant_name, seed)
                model_done = ckpt is not None and ckpt.exists()

                if csv_done and (model_done or MODELS_DIR is None):
                    print(f"{label}  SKIP")
                    continue

                print(f"{label} ...", end="", flush=True)

                try:
                    model      = build_catboost_model(variant_cfg, data["task_type"], seed)
                    train_time = train_model(model, data)
                    score      = compute_score(model, data)
                    n_trees    = model.tree_count_
                    save_model(model, ds_name, variant_name, seed)
                    del model
                except Exception as exc:
                    print(f"\n    ✗ ERROR: {exc}")
                    append_row({
                        "source":       source,
                        "dataset":      ds_name,
                        "variant":      variant_name,
                        "seed":         seed,
                        "task_type":    data["task_type"],
                        "metric":       metric_name,
                        "score":        float("nan"),
                        "n_trees":      -1,
                        "train_time_s": float("nan"),
                    })
                    gc.collect()
                    continue

                gc.collect()
                seed_scores.append(score)
                append_row({
                    "source":       source,
                    "dataset":      ds_name,
                    "variant":      variant_name,
                    "seed":         seed,
                    "task_type":    data["task_type"],
                    "metric":       metric_name,
                    "score":        round(score, 8),
                    "n_trees":      n_trees,
                    "train_time_s": round(train_time, 2),
                })
                print(f"  score={score:.4f}  trees={n_trees:5d}  t={train_time:.1f}s")

            if seed_scores:
                print(f"  {'':20s}       → mean={np.mean(seed_scores):.4f} "
                      f"± {np.std(seed_scores):.4f}")

        del data
        gc.collect()

# =============================================================================
# SUMMARY
# =============================================================================

def print_summary() -> None:
    if not RESULTS_CSV_LOCAL.exists():
        return
    df = pd.read_csv(RESULTS_CSV_LOCAL).dropna(subset=["score"])
    print("\n=== SUMMARY ===")
    for source in df["source"].unique():
        sub     = df[df["source"] == source]
        summary = (
            sub.groupby(["dataset", "variant"])["score"]
            .mean().unstack("variant").round(8)
        )
        print(f"\n[{source.upper()}]")
        print(summary.to_string())

# =============================================================================
# OPENML DATASET REGISTRY
# =============================================================================

OPENML_DATASETS: dict[str, dict] = {
    "superconductivity":       {"id": 43174, "target": "critical_temp"},
    "solar_flare":             {"id": 44973, "target": "c_class_flares"},
    "student_performance_por": {"id": 44974, "target": "G3"},
    "QSAR_fish_toxicity":      {"id": 44976, "target": "LC50"},
    "video_transcoding":       {"id": 44980, "target": "utime"},
    "wave_energy":             {"id": 44981, "target": "energy_total"},
    "sarcos":                  {"id": 44976, "target": "V22"},
    "california_housing":      {"id": 43939, "target": "medianHouseValue"},
    "diamonds":                {"id": 42225, "target": "price"},
    "kin8nm":                  {"id": 189,   "target": "y"},
    "pumadyn32nh":             {"id": 308,   "target": "thetadd6"},
    "miami_housing":           {"id": 43093, "target": "SALE_PRC"},
    "space_ga":                {"id": 44983, "target": "ln_votes_pop"},
    "socmob":                  {"id": 44984, "target": "counts_for_sons_current_occupation"},
    "kings_county":            {"id": 42731, "target": "price"},
    "health_insurance":        {"id": 44986, "target": "whrswk"},
    "facebook_comment_volume": {"id": 4549,  "target": None},
    "airline_delay":           {"id": 1169,  "target": None},
}

# =============================================================================
# ENTRY POINT
# =============================================================================

if __name__ == "__main__":

    # 1. Подключиться к Google Drive через rclone
    GDRIVE_READY = _setup_rclone()

    # 2. Скачать существующий CSV с Drive (чтобы скипать уже сделанное)
    if GDRIVE_READY:
        _pull_csv_from_drive()

    # 3. Запустить бенчмарки
    run_benchmark(OPENML_DATASETS, source="openml")

    if TABM_ROOT.exists():
        tabm_datasets = {
            d.name: None
            for d in sorted(TABM_ROOT.iterdir())
            if d.is_dir() and d.name not in TABM_SKIP and (d / "info.json").exists()
        }
        run_benchmark(tabm_datasets, source="tabm")
    else:
        print(f"\nTabM root not found: {TABM_ROOT} — skipping.")

    print_summary()

  rclone.conf written to /root/.config/rclone/rclone.conf
  rclone not found — installing...
  rclone installed.
✓  rclone connected to remote 'gdrive'
✓  Pulled existing CSV from Drive (1530 rows — will skip these).

Source: OPENML  |  datasets=18  variants=3  seeds=15

──────────────────────────────────────────────────────────────────────
Dataset: superconductivity

>>> [OpenML] Loading superconductivity...
    Shape — train: (17010, 81)  val: (2126, 81)  test: (2127, 81)
  Task: regression  |  Metric: R2
  Shapes — train: (17010, 81)  val: (2126, 81)  test: (2127, 81)
  [   1/810] CB_Default           seed= 0  SKIP
  [   2/810] CB_Default           seed= 1  SKIP
  [   3/810] CB_Default           seed= 2  SKIP
  [   4/810] CB_Default           seed= 3  SKIP
  [   5/810] CB_Default           seed= 4  SKIP
  [   6/810] CB_Default           seed= 5  SKIP
  [   7/810] CB_Default           seed= 6  SKIP
  [   8/810] CB_Default           seed= 7  SKIP
  [   9/810] CB_Default           seed